# Model 6: Pairwise Identity Verification (PIV)

PIV tests whether a small MLP can distinguish same-animal pairs from different-animal pairs using the supplied MIEW-ID descriptors. The first experiment measures **known-identity matching only**. It compares PIV with raw cosine and LDA cosine on the same gallery/query episodes. It does not estimate `new_individual` or produce a Kaggle submission.

Identities are divided into fit (60%), development (20%) and final evaluation (20%) roles before any pairs are sampled. Near-duplicate groups are held together when making queries. PCA, LDA, scaling and MLP are fitted with fit-role data only. The method and evaluation protocol are documented in the embedded README.

No wildlife-data score is recorded in this notebook. A gain is a hypothesis to test. The model uses CPU; GPU acceleration is not enabled by scikit-learn's MLP.


## 1. Dependencies and source snapshot
The code cell writes the tested script and documentation into a writable run directory. The required packages are NumPy, pandas, SciPy, scikit-learn and joblib.


In [ ]:
from pathlib import Path
from datetime import datetime
import sys, subprocess, json

CANDIDATES = [Path.cwd(), Path.cwd() / "Models" / "Abhay" / "Model6", Path.cwd() / "Model6"]
PROJECT_DIR = next((p for p in CANDIDATES if (p / "matching.py").is_file()), Path.cwd())
RUN_DIR = PROJECT_DIR / "local_runs" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
SOURCE_DIR = RUN_DIR / "source"
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
(SOURCE_DIR / "matching.py").write_text('"""Model 6: pairwise identity verification on supplied MIEW-ID descriptors."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import csr_matrix\nfrom scipy.sparse.csgraph import connected_components\nfrom sklearn.decomposition import PCA\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis\nfrom sklearn.neural_network import MLPClassifier\nfrom sklearn.preprocessing import StandardScaler\n\nVIEWS = {"left": -1, "frontleft": -1, "backleft": -1,\n         "right": 1, "frontright": 1, "backright": 1}\n\n\ndef unit(a):\n    return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)\n\n\ndef load(data):\n    meta = pd.read_csv(data / "train.csv")\n    index = pd.read_csv(data / "miewid_ids.csv")\n    if not meta.id.is_unique or not index.id.is_unique:\n        raise ValueError("Image IDs must be unique")\n    ix = pd.Index(index.id).get_indexer(meta.id)\n    if np.any(ix < 0):\n        raise ValueError("Training image missing from descriptor ID mapping")\n    raw = np.load(data / "miewid_features.npy", mmap_mode="r")\n    if len(raw) != len(index):\n        raise ValueError("Descriptor and ID table lengths differ")\n    x = np.asarray(raw[ix], dtype=np.float32)\n    if not np.isfinite(x).all() or (np.linalg.norm(x, axis=1) == 0).any():\n        raise ValueError("Invalid descriptor")\n    if meta.identity.isna().any() or not set(meta.species) <= {"hyena", "leopard"}:\n        raise ValueError("Invalid training identity or species")\n    return meta.reset_index(drop=True), unit(x)\n\n\ndef split_identities(meta, seed):\n    rng = np.random.default_rng(seed)\n    roles = np.empty(len(meta), dtype=object)\n    for species in ("hyena", "leopard"):\n        names = np.array(sorted(meta.loc[meta.species == species, "identity"].unique()))\n        rng.shuffle(names)\n        a, b = int(.6 * len(names)), int(.8 * len(names))\n        for role, block in (("fit", names[:a]), ("development", names[a:b]),\n                            ("evaluation", names[b:])):\n            roles[(meta.species == species) & meta.identity.isin(block)] = role\n    assert pd.DataFrame({"identity": meta.identity, "role": roles}).groupby("identity").role.nunique().max() == 1\n    return roles\n\n\ndef embedding(x, fit_idx, seed, dimensions):\n    n = min(dimensions, len(fit_idx) - 1, x.shape[1])\n    if n < 2:\n        raise ValueError("Too few fitting images")\n    pca = PCA(n_components=n, svd_solver="randomized", random_state=seed).fit(x[fit_idx])\n    z = unit(pca.transform(x).astype(np.float32))\n    return z, pca\n\n\ndef groups(meta, x, indices, threshold):\n    """Near-duplicate groups within an identity, using supplied global features only."""\n    labels = np.full(len(meta), -1, dtype=int)\n    group_id = 0\n    for _, d in meta.iloc[indices].groupby("identity", sort=True):\n        ix = d.index.to_numpy()\n        links = x[ix] @ x[ix].T >= threshold\n        local = connected_components(csr_matrix(links), directed=False)[1]\n        for g in np.unique(local):\n            labels[ix[local == g]] = group_id\n            group_id += 1\n    return labels\n\n\ndef gallery_query(meta, indices, group_ids, seed):\n    rng = np.random.default_rng(seed)\n    gallery, query = [], []\n    for _, d in meta.iloc[indices].groupby("identity", sort=True):\n        ix = d.index.to_numpy()\n        gs = np.unique(group_ids[ix])\n        if len(gs) < 2:\n            gallery.extend(ix)\n            continue\n        n_query_groups = min(max(1, int(round(.3 * len(gs)))), 8, len(gs) - 1)\n        chosen = rng.choice(gs, size=n_query_groups, replace=False)\n        query.extend(ix[np.isin(group_ids[ix], chosen)])\n        gallery.extend(ix[~np.isin(group_ids[ix], chosen)])\n    gallery, query = np.array(gallery, dtype=int), np.array(query, dtype=int)\n    if not len(query):\n        raise ValueError("No query groups available")\n    assert set(meta.identity.iloc[query]) <= set(meta.identity.iloc[gallery])\n    assert not set(group_ids[query]) & set(group_ids[gallery])\n    return gallery, query\n\n\ndef train_pairs(meta, x, z, fit_idx, group_ids, seed, hard_k=30, per_image=2):\n    """One positive, one hard negative and one random negative per anchor/repeat."""\n    rng = np.random.default_rng(seed)\n    y = meta.identity.to_numpy()\n    sp = meta.species.to_numpy()\n    left, right, labels = [], [], []\n    for species in ("hyena", "leopard"):\n        ix = fit_idx[sp[fit_idx] == species]\n        sim = x[ix] @ x[ix].T\n        for j, anchor in enumerate(ix):\n            positives = ix[(y[ix] == y[anchor]) & (group_ids[ix] != group_ids[anchor])]\n            negatives = ix[y[ix] != y[anchor]]\n            if not len(positives) or not len(negatives):\n                continue\n            candidates = np.argsort(-sim[j])\n            hard = ix[candidates[y[ix[candidates]] != y[anchor]][:hard_k]]\n            if not len(hard):\n                raise ValueError("No hard negatives")\n            for _ in range(per_image):\n                # Positive is biased toward a different flank when available.\n                opposite = positives[np.array([VIEWS.get(meta.viewpoint.iloc[p], 0) == -VIEWS.get(meta.viewpoint.iloc[anchor], 0)\n                                               and VIEWS.get(meta.viewpoint.iloc[anchor], 0) != 0 for p in positives])]\n                pos = rng.choice(opposite if len(opposite) else positives)\n                for candidate, match in ((pos, 1), (rng.choice(hard), 0), (rng.choice(negatives), 0)):\n                    left.append(anchor); right.append(candidate); labels.append(match)\n    a, b = np.array(left), np.array(right)\n    if len(np.unique(labels)) != 2:\n        raise ValueError("Pair set lacks both classes")\n    return pair_features(meta, x, z, a, b), np.array(labels)\n\n\ndef pair_features(meta, x, z, a, b):\n    """Symmetric pair representation; invariant to pair order."""\n    va = np.array([VIEWS.get(v, 0) for v in meta.viewpoint.iloc[a]], dtype=np.int8)\n    vb = np.array([VIEWS.get(v, 0) for v in meta.viewpoint.iloc[b]], dtype=np.int8)\n    side = np.column_stack(((va != 0) & (vb != 0) & (va == vb), va * vb == -1, (va == 0) | (vb == 0)))\n    return np.column_stack((np.abs(z[a] - z[b]), z[a] * z[b],\n                            np.sum(x[a] * x[b], axis=1), np.sum(z[a] * z[b], axis=1), side)).astype(np.float32)\n\n\ndef fit_lda(meta, z, fit_idx):\n    models = {}\n    for species in ("hyena", "leopard"):\n        ix = fit_idx[meta.species.iloc[fit_idx].to_numpy() == species]\n        model = LinearDiscriminantAnalysis(solver="eigen", shrinkage=.3).fit(z[ix], meta.identity.iloc[ix])\n        models[species] = model\n    return models\n\n\ndef lda_transform(meta, z, models):\n    out = {}\n    for species, model in models.items():\n        ix = np.flatnonzero(meta.species.to_numpy() == species)\n        out[species] = (ix, unit(model.transform(z[ix]).astype(np.float32)))\n    return out\n\n\ndef evaluate(meta, x, z, lda, network, scaler, gallery, queries, batch=4096):\n    results = []\n    truth = meta.identity.to_numpy()\n    species = meta.species.to_numpy()\n    for sp in ("hyena", "leopard"):\n        gi = gallery[species[gallery] == sp]\n        qi = queries[species[queries] == sp]\n        if not len(qi):\n            continue\n        names, inv = np.unique(truth[gi], return_inverse=True)\n        lid, lz = lda[sp]\n        loc = np.full(len(meta), -1, dtype=int)\n        loc[lid] = np.arange(len(lid))\n        raw = x[qi] @ x[gi].T\n        projected = lz[loc[qi]] @ lz[loc[gi]].T\n        for row, q in enumerate(qi):\n            pair = np.empty(len(gi), dtype=np.float32)\n            for start in range(0, len(gi), batch):\n                g = gi[start:start + batch]\n                features = pair_features(meta, x, z, np.full(len(g), q, dtype=int), g)\n                pair[start:start + len(g)] = network.predict_proba(scaler.transform(features))[:, 1]\n            row_scores = {"cosine": raw[row], "lda": projected[row], "pair_mlp": pair}\n            record = {"id": meta.id.iloc[q], "species": sp, "identity": truth[q],\n                      "viewpoint": meta.viewpoint.iloc[q]}\n            for method, scores in row_scores.items():\n                per_identity = np.full(len(names), -np.inf)\n                np.maximum.at(per_identity, inv, scores)\n                ranked = names[np.argsort(-per_identity, kind="stable")[:5]]\n                rank = np.flatnonzero(ranked == truth[q])\n                record[f"{method}_rr5"] = 1 / (rank[0] + 1) if len(rank) else 0.\n                record[f"{method}_rank1"] = int(bool(len(rank) and rank[0] == 0))\n                record[f"{method}_top5"] = " ".join(ranked)\n                record[f"{method}_best_wrong"] = float(np.max(per_identity[names != truth[q]]))\n                record[f"{method}_correct"] = float(per_identity[names == truth[q]][0])\n            results.append(record)\n    return pd.DataFrame(results)\n\n\ndef summary(frame):\n    return {m: {"map5": float(frame[f"{m}_rr5"].mean()),\n                "top1": float(frame[f"{m}_rank1"].mean()),\n                "top5": float((frame[f"{m}_rr5"] > 0).mean()),\n                "wrong_identity_beats_true": float((frame[f"{m}_best_wrong"] >= frame[f"{m}_correct"]).mean())}\n            for m in ("cosine", "lda", "pair_mlp")}\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__)\n    ap.add_argument("--data", type=Path, required=True)\n    ap.add_argument("--output", type=Path, required=True)\n    ap.add_argument("--seed", type=int, default=2026)\n    ap.add_argument("--dimensions", type=int, default=128)\n    ap.add_argument("--duplicate-cosine", type=float, default=.97)\n    ap.add_argument("--hard-k", type=int, default=30)\n    ap.add_argument("--pairs-per-image", type=int, default=2)\n    ap.add_argument("--max-iter", type=int, default=80)\n    args = ap.parse_args()\n    if not 0 < args.duplicate_cosine < 1 or min(args.dimensions, args.hard_k, args.pairs_per_image, args.max_iter) < 1:\n        ap.error("Invalid positive counts or duplicate threshold")\n    args.output.mkdir(parents=True, exist_ok=True)\n    meta, x = load(args.data)\n    roles = split_identities(meta, args.seed)\n    pd.DataFrame({"id": meta.id, "species": meta.species, "identity": meta.identity,\n                  "role": roles}).to_csv(args.output / "identity_roles.csv", index=False)\n    fit = np.flatnonzero(roles == "fit")\n    z, pca = embedding(x, fit, args.seed, args.dimensions)\n    groups_fit = groups(meta, x, fit, args.duplicate_cosine)\n    features, targets = train_pairs(meta, x, z, fit, groups_fit, args.seed,\n                                    args.hard_k, args.pairs_per_image)\n    print(f"Fitting pair verifier on {len(features)} pairs", flush=True)\n    scaler = StandardScaler().fit(features)\n    network = MLPClassifier(hidden_layer_sizes=(64, 32), alpha=.01, max_iter=args.max_iter,\n                            early_stopping=False, random_state=args.seed, n_iter_no_change=12,\n                            batch_size=256, learning_rate_init=.001)\n    network.fit(scaler.transform(features), targets)\n    lda = lda_transform(meta, z, fit_lda(meta, z, fit))\n    metrics = {"config": {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()},\n               "pair_count": len(features), "fit_identities": int(meta.identity.iloc[fit].nunique()),\n               "mlp_epochs": int(network.n_iter_)}\n    for role in ("development", "evaluation"):\n        ix = np.flatnonzero(roles == role)\n        group_ids = groups(meta, x, ix, args.duplicate_cosine)\n        gi, qi = gallery_query(meta, ix, group_ids, args.seed + (1 if role == "development" else 2))\n        used = np.r_[gi, qi]\n        pd.DataFrame({"id": meta.id.iloc[used].to_numpy(),\n                      "species": meta.species.iloc[used].to_numpy(),\n                      "identity": meta.identity.iloc[used].to_numpy(),\n                      "duplicate_group": group_ids[used],\n                      "assignment": ["gallery"] * len(gi) + ["query"] * len(qi)}).to_csv(\n                          args.output / f"{role}_split.csv", index=False)\n        frame = evaluate(meta, x, z, lda, network, scaler, gi, qi)\n        frame.to_csv(args.output / f"{role}_predictions.csv", index=False)\n        metrics[role] = {"queries": len(frame), "identities": int(meta.identity.iloc[ix].nunique()),\n                         "query_identities": int(meta.identity.iloc[qi].nunique()),\n                         "gallery_only_identities": int(meta.identity.iloc[gi].nunique() - meta.identity.iloc[qi].nunique()),\n                         "scores": summary(frame),\n                         "by_species": {sp: summary(part) for sp, part in frame.groupby("species")}}\n        print(role, json.dumps(metrics[role]), flush=True)\n    joblib.dump({"pca": pca, "lda": lda, "scaler": scaler, "network": network}, args.output / "model.joblib")\n    (args.output / "metrics.json").write_text(json.dumps(metrics, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
(SOURCE_DIR / "README.md").write_text('# Model 6: Pairwise Identity Verification (PIV)\n\nPIV tests whether a small MLP trained on pairs of supplied MIEW-ID descriptors can improve matching between **known** animal identities. This is a separate matching experiment. It does not perform open-set `new_individual` detection or create a competition submission. A gain in this experiment would justify a subsequent open-set design; it is not itself evidence of a higher Kaggle score.\n\nThe first identity-disjoint test did **not** pass the matching gate: evaluation MAP@5 was 0.3046 for PIV versus 0.9091 for original cosine. See `RESULTS.md` for the full comparison and failure analysis. The implementation is retained as a reproducible negative result.\n\n## Method\n\n1. Align descriptors to `train.csv` through `miewid_ids.csv`, then normalize each vector.\n2. Split identities separately for hyenas and leopards: 60% fit, 20% development, 20% final evaluation. The split occurs before pair generation. No identity spans roles.\n3. Fit a 128-dimensional PCA on fit-role images only. Normalize projected vectors. Fit species-specific LDA baselines on the same fit identities.\n4. Form positive pairs from different near-duplicate groups of the same identity, preferring opposite flanks when available. Form hard negatives from other identities among the 30 most similar same-species images, plus random negatives. Images whose identity has only one group do not supply positive training pairs. This prevents near-identical sightings from making the task artificially easy.\n5. Pair features are `|z_a-z_b|`, `z_a*z_b`, original cosine, projected cosine, and three simple viewpoint indicators. They are symmetric in pair order. Features are scaled using fit-role pairs only.\n6. Fit an MLP with hidden layers (64, 32), L2 penalty 0.01 and fixed seed. Early stopping is disabled because a random pair-level validation split could share identities with training. Development identities are used to assess the fixed configuration; no automated hyperparameter search runs on them.\n7. For each development and evaluation identity, hold out about 30% of whole near-duplicate groups as query sightings, capped at eight groups while retaining at least one gallery group. Identities with a single group remain gallery-only. Compare each query to all same-species gallery images. The maximum score per identity produces the ranking.\n\nCosine similarity, LDA cosine similarity, and PIV probability use identical gallery/query splits and identity aggregation. Report MAP@5, top-one accuracy, top-five recall and the fraction of queries for which a wrong identity scores at least as high as the correct one. Species scores and per-query predictions are exported.\n\nThe near-duplicate grouping uses only supplied global descriptor cosine similarity >=0.97 within an identity. It approximates encounter grouping; original encounter IDs are unavailable. The threshold is fixed before evaluation. On the supplied training set, this produces 215 development queries from 92 identities and 271 evaluation queries from 89 identities (seed 2026); the other identities have only one group and remain gallery-only. Gallery/query evaluation is *closed-set*: every evaluated query has a matching identity in its gallery. This isolates the proposed matching improvement. Positive-pair sampling excludes single-group identities and may underrepresent very small identities; this is a limitation to inspect in the results.\n\nPair examples are dependent: thousands of sampled pairs do not constitute thousands of independent animals. The evaluation identities, not pair rows, are held out. A single split still has variance; repeated predeclared identity splits would be needed before claiming a reliable improvement. The source training set has no original encounter IDs, so near-duplicate separation is approximate.\n\n## Running\n\nKaggle: upload `PIV_Kaggle.ipynb`, attach the existing `projectwild3` competition input, and run the cells in order. Only `train.csv`, `miewid_ids.csv`, and `miewid_features.npy` are used. SIFT files and raw images are not needed for this first test. CPU is sufficient; scikit-learn\'s MLP has no GPU support.\n\nWindows: open `PIV_Local.ipynb` in JupyterLab or VS Code. The dataset defaults to `Path.home()/"Downloads"/"reid_student_data_main"`. The notebook uses the active Python kernel and writes a new local run directory. Install dependencies from `requirements.txt` if the kernel lacks them.\n\nCLI, from this directory:\n\n```powershell\npython -m pip install -r requirements.txt\npython matching.py --data "$env:USERPROFILE\\Downloads\\reid_student_data_main" --output ./local_runs/piv_run\n```\n\nResults are written to `metrics.json`, `identity_roles.csv`, `development_split.csv`, `evaluation_split.csv`, per-query prediction CSVs and `model.joblib`. The split CSVs include identity, duplicate-group ID and gallery/query assignment so the grouping can be audited. These runtime artifacts are excluded from Git. Progress prints the pair count, model fit, and both score summaries. `--dimensions`, `--hard-k`, `--pairs-per-image`, and `--max-iter` permit development experiments; a final performance claim requires an evaluation split not used to select them. Run `python -m unittest test_matching.py` for the small synthetic pipeline and split-integrity checks.\n\n## Decision rule\n\nProceed to open-set modelling only if PIV improves known-animal MAP@5 and top-one identity accuracy on evaluation identities, ideally for both species, with stable results across additional predeclared splits. If the improvement appears only on development identities or only on a small slice, the result does not support a more complex full pipeline. A promising verifier could then be combined with a separately validated `new_individual` detector; that second stage is outside Model 6\'s current scope.\n\n## Competition rules and attribution\n\nThe project statement permits MLPs trained from scratch on supplied features, PCA, LDA and distance-based retrieval. PIV trains only on provided feature vectors and labels. It uses no pretrained models, images or external data. Dalvin\'s retrieval work motivated the comparison, but this implementation has separate pair construction, pair features, MLP training and evaluation. No baseline code is copied into Model 6.\n', encoding="utf-8")
print("Source snapshot:", SOURCE_DIR)


## 2. Local data and configuration
The default data directory is `Path.home()/"Downloads"/"reid_student_data_main"`. A timestamped run directory preserves earlier results. For an isolated Python environment, use `python -m pip install -r requirements.txt` from this model directory, then select that interpreter as the notebook kernel.


In [ ]:
from importlib.util import find_spec
missing = [n for n in ("numpy", "pandas", "scipy", "sklearn", "joblib") if find_spec(n) is None]
if missing:
    raise ImportError(f"Missing packages in {sys.executable}: {missing}. Install requirements and restart the kernel.")
DATA = Path.home() / "Downloads" / "reid_student_data_main"
OUTPUT = RUN_DIR / "results"
for name in ("train.csv", "miewid_ids.csv", "miewid_features.npy"):
    if not (DATA / name).is_file():
        raise FileNotFoundError(DATA / name)
SEED = 2026
DIMENSIONS = 128
DUPLICATE_COSINE = 0.97
HARD_K = 30
PAIRS_PER_IMAGE = 2
MAX_ITER = 80
print("Data:", DATA)
print("Output:", OUTPUT)


## 3. Train and evaluate
The fit role creates positive, hard-negative and random-negative pairs. The development and evaluation roles remain identity-disjoint from fitting and from each other. The script writes reproducible identity and gallery/query split files. The result does not include open-set detection.


In [ ]:
command = [sys.executable, str(SOURCE_DIR / "matching.py"),
           "--data", str(DATA), "--output", str(OUTPUT),
           "--seed", str(SEED), "--dimensions", str(DIMENSIONS),
           "--duplicate-cosine", str(DUPLICATE_COSINE),
           "--hard-k", str(HARD_K), "--pairs-per-image", str(PAIRS_PER_IMAGE),
           "--max-iter", str(MAX_ITER)]
print(" ".join(command))
subprocess.run(command, check=True)


## 4. Compare methods
The decision gate is known-identity MAP@5 and top-one accuracy on evaluation identities, checked separately for hyenas and leopards. Scores from the development role are diagnostic; they do not justify tuning against the final evaluation. No leaderboard improvement is inferred from this test alone.


In [ ]:
import pandas as pd
metrics = json.loads((OUTPUT / "metrics.json").read_text())
for role in ("development", "evaluation"):
    print(role, "queries:", metrics[role]["queries"], "identities:", metrics[role]["identities"])
    display(pd.DataFrame(metrics[role]["scores"]).T)
    for species, scores in metrics[role]["by_species"].items():
        print(role, species)
        display(pd.DataFrame(scores).T)
    rows = pd.read_csv(OUTPUT / f"{role}_predictions.csv")
    rows["piv_minus_cosine"] = rows.pair_mlp_rr5 - rows.cosine_rr5
    display(rows.sort_values("piv_minus_cosine").head(10))


## 5. Decision and artifacts
A result table and per-query predictions are saved in the run directory. If PIV does not improve the fixed evaluation, stop before extending it to `new_individual` ranking. If it improves consistently, validate on additional predeclared identity splits before building a complete open-set method.


In [ ]:
evaluation = metrics["evaluation"]["scores"]
for baseline in ("cosine", "lda"):
    print(f"PIV minus {baseline} MAP@5: {evaluation['pair_mlp']['map5'] - evaluation[baseline]['map5']:+.4f}")
    print(f"PIV minus {baseline} top-one: {evaluation['pair_mlp']['top1'] - evaluation[baseline]['top1']:+.4f}")
if all(evaluation["pair_mlp"][metric] > evaluation["cosine"][metric]
       for metric in ("map5", "top1")):
    print("Matching gate passed; inspect species results and replicate before open-set work.")
else:
    print("Matching gate failed; pair verifier is not ready for open-set scoring.")
print("Metrics:", OUTPUT / "metrics.json")
print("Evaluation predictions:", OUTPUT / "evaluation_predictions.csv")
print("Model:", OUTPUT / "model.joblib")
